In [ ]:
### Load libraries
library(NMF)
library(Seurat)
library(CellChat)
library(dplyr)

### Set directories
mainDir <- "/data/ebaird/scRNAseq/SCENTINELsep24/"
repDir <- paste0(mainDir, "cellchat_neurotransmitters/")
figDir <- paste0(repDir, "figs/")
tabDir <- paste0(repDir, "tables/")
refsDir <- "/data/ebaird/scRNAseq/refs/"

dir.create(repDir, recursive = TRUE, showWarnings = FALSE)
dir.create(figDir, recursive = TRUE, showWarnings = FALSE)
dir.create(tabDir, recursive = TRUE, showWarnings = FALSE)
dir.create(refsDir, recursive = TRUE, showWarnings = FALSE)

### Set colours
mycols <- c(1, '#ffffe5','#fff7bc','#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506')
mycols11 <- c(1, '#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506', "purple", "violet", "gray")
mycols13 <- c(1, '#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506', "purple", "violet", "gray", "blue", "green")
mycols17 <- c(1, '#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506', "purple", "violet", "gray", "blue", "green", rainbow(4))

mycols20 <- c("yellow", '#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506', "purple", "violet", "chartreuse", "blue", "green", rainbow(4), "darkslategray3", "darksalmon", "darkorchid4", "cyan")

corner <- function(x) x[1:5,1:5]
cols <- c(colorRamps::matlab.like2(20)[1:18], "deeppink2", "deeppink3", "deeppink4")

getdensity <- function(x, y, ...) {
      dens <- MASS::kde2d(x, y, ...)
      ix <- findInterval(x, dens$x)
      iy <- findInterval(y, dens$y)
      ii <- cbind(ix, iy)
      return(dens$z[ii])
}

In [ ]:
seurat_object <- readRDS(paste0(mainDir, "QC_clustering/merged_clusters.rds"))

In [ ]:
lr_pairs <- data.frame(
  ligand = c(
    # Acetylcholine (ACh) system
    "ChAT",          "VAChT",
    # GABA system
    "Gad1",          "VGAT",
    # Glutamate system
    "VGlut",
    # Dopamine system
    "ple",            "DAT",        "Vmat",
    # Octopamine / Tyramine system
    "Tdc2",          "Tbh",
    # Serotonin (5-HT) system
    "Trh",           "Hn",        "Vmat"
  ),
  receptor = c(
    # ACh receptors — nicotinic (ionotropic) & muscarinic (metabotropic)
    "nAChRα1", "nAChRβ1",
    "Rdl",      "Rdl",   # placeholder GABA‑A; see notes below
    "mGluR",  # glutamate metabotropic (mGluR)
    "Dop1R1",   "Dop1R2",   "Dop2R",
    "Oamb",   "Oct-TyrR",
    "5-HT1A",   "5-HT2A",    "5-HT7"
  ),
  pathway = c(
    "Acetylcholine", "Acetylcholine", 
    "GABA",          "GABA", 
    "Glutamate", 
    "Dopamine",     "Dopamine",   "Dopamine", 
    "Octopamine",   "Octopamine",
    "Serotonin",    "Serotonin",   "Serotonin"
  ),
  stringsAsFactors = FALSE
)

lr_pairs <- lr_pairs[
  lr_pairs$ligand %in% rownames(seurat_object) &
  lr_pairs$receptor %in% rownames(seurat_object),
]


In [ ]:
# Save and inspect
write.csv(lr_pairs, paste0(refsDir, "custom_fly_neuro_LR.csv"), row.names=FALSE)
print(lr_pairs)

In [ ]:
# Add a 'c' before each seurat_clusters meta.data entry to make CellChat happy
seurat_object$seurat_clusters <- paste0("c", seurat_object$seurat_clusters)
Idents(seurat_object) <- seurat_object$seurat_clusters

In [ ]:
# 4) Prepare data for CellChat (convert Seurat to CellChat object)
data.input <- GetAssayData(seurat_object, assay = "RNA", slot = "data")
data.input <- as.matrix(data.input)
meta <- seurat_object@meta.data

cellchat <- createCellChat(object = data.input, meta = meta, group.by = "seurat_clusters")
# replace default DB with custom (basic example)
db_interaction <- data.frame(
  ligand = lr_pairs$ligand,
  receptor = lr_pairs$receptor,
  complex = NA,  # optional
  pathway = lr_pairs$pathway,  # optional but recommended
  stringsAsFactors = FALSE
)

customDB <- list(
  interaction = db_interaction,
  complex = data.frame(),   # must exist even if empty
  cofactor = data.frame()   # can remain empty
)

# CellChat expects a specific format; easiest is to create a simple DB object:
cellchat@DB <- customDB
# Alternatively, add your custom pairs to cellchatDB manually following CellChat docs.

In [ ]:
head(data.input)

In [ ]:
# 5) Run pipeline (filtering, compute communication probability)
cellchat <- subsetData(cellchat) # default filter
dim(cellchat@data.signaling)
head(rownames(cellchat@data.signaling))
cellchat <- identifyOverExpressedGenes(cellchat)
cellchat <- identifyOverExpressedInteractions(cellchat)
cellchat <- computeCommunProb(cellchat)
cellchat <- filterCommunication(cellchat, min.cells = 5)
cellchat <- computeCommunProbPathway(cellchat)
cellchat <- aggregateNet(cellchat)

In [ ]:
# Example: check which ligand/receptor genes are in your Seurat object
intersect(lr_pairs$ligand, rownames(data.input))
intersect(lr_pairs$receptor, rownames(data.input))


In [ ]:
class(data.input)
dim(data.input)
head(rownames(data.input))

In [ ]:
lr_genes <- unique(c(lr_pairs$ligand, lr_pairs$receptor))
present <- lr_genes %in% rownames(data.input)
data.frame(gene = lr_genes, in_data = present)
